# Two-Arm Bandit: Preprocessed Data Exploration

Use the bundled loader and visualizers to inspect a session: find a session directory with `metrics.json`, load neuronal/behavioral matrices, and render quick sanity-check plots.

In [ ]:
from pathlib import Path
import numpy as np
from sklearn.model_selection import KFold

from ncmcm.data_loaders.bandit_task import BanditTaskNeuroPixelsDataset
from ncmcm.visualisers.behavioural_discrete import (
    plot_behavior_state_lengths_boxplot,
    plot_behavior_state_sample_frequencies_barchart,
)
from ncmcm.visualisers.twoArmBandit_session_visualization import (
    generate_interactive_session_plot,
)
from ncmcm.visualisers.neuronal_behavioural import plotting_neuronal_behavioural_plotly

# Root the search at the folder containing bandit sessions
# (expected to contain per-session subfolders with metrics.json and spike files)
data_root = Path("/home/kerim/Projects/Neural Algorithms/NC-MCM/datasets/raw/twoArmBandit")

data_root


In [ ]:
# Find session folders containing metrics.json
session_dirs = sorted({p.parent for p in data_root.glob("**/metrics.json")})
if not session_dirs:
    raise FileNotFoundError(f"No sessions with metrics.json found under {data_root}")

# Pick the first session by default; change index to select another
session_path = session_dirs[0]
session_path

In [ ]:
# Load neuronal and behavioral time-series
# Adjust downsample_fs to control resolution (None keeps native)
dataset = BanditTaskNeuroPixelsDataset(
    data_path=session_path,
    downsample_fs=30,
    downsample_method="count",
    good_neurons_only=True,
    state_transitions=None,
    normalize_method=None,
    choosing_state_mode="side",
    b_mode="decision_strict", 
    # recompute_cache=True
)

# Quick shapes sanity check
{
    "neuronal_shape": dataset.x.shape,
    "behavior_shape": dataset.b.shape,
    "fs": dataset.fs,
    "states": dataset.b_labels_dict,
}


In [ ]:
# Basic behavioral stats
b_dense = dataset.b.toarray().flatten()
unique_states, counts = np.unique(b_dense, return_counts=True)
state_counts = {
    dataset.b_labels_dict.get(int(s), str(s)): int(c) for s, c in zip(unique_states, counts)
}

{
    "n_timepoints": int(dataset.b.shape[1]),
    "n_neurons": int(dataset.x.shape[0]),
    "state_counts": state_counts,
}

In [ ]:
# Plot state frequency distribution and segment length distributions
freq_fig = plot_behavior_state_sample_frequencies_barchart(
    dataset.b,
    dataset.b_labels_dict,
    show_fig=True,
    show_percentages=True,
    show_counts=True,
    color_map=dataset.get_color_map_for_plotting(),
    title="Behavioral state frequency",
)

length_fig = plot_behavior_state_lengths_boxplot(
    dataset.b,
    dataset.b_labels_dict,
    sampling_frequency=dataset.fs,
    convert_to_seconds=True,
    color_map=dataset.get_color_map_for_plotting(),
    title="Behavioral state segment lengths (s)",
)

In [ ]:
# Neuronal + behavioral visualization using built-in helper
x_dense = dataset.x.toarray().T
b_dense = dataset.b.toarray().flatten()

color_map = dataset.get_color_map_for_plotting()
fig_nb = plotting_neuronal_behavioural_plotly(
    x_dense,
    b=b_dense,
    b_names=dataset.b_labels_dict,
    b_colors=color_map,
    show_fig=True,
    colorscale="Viridis",
)


## Train / Test Split Visualisation

Reproduce the neuronal–behavioural heatmap from the grid-search script and overlay:
- **Green vertical lines** marking reward-block boundaries (Better-Left / Better-Right).
- **Red dotted vertical lines** marking the start and end of the held-out test split (fold 4 of 7, same as the grid-search default).


In [ ]:
# Parameters matching the grid-search script defaults
split_window = 50  # matches grid-search default

# Output
output_html = Path("results/neuronal_behavioural_with_test_split.html")
output_html.parent.mkdir(parents=True, exist_ok=True)

# Reuse the dataset loaded above
x_split = dataset.x.T.toarray().astype(np.float32)
b_split = dataset.b.toarray().flatten()
b_labels_split = dataset.b_labels
b_colors_split = dataset.get_color_map_for_plotting()

# Build block-boundary markers (green lines)
block_labels = np.array(dataset.block_labels, dtype=object)

def _block_annotation_text(label):
    if label is None:
        return "Block"
    label_lower = str(label).lower()
    if "left" in label_lower:
        return "Better L"
    if "right" in label_lower:
        return "Better R"
    return str(label)

block_markers = []
previous_label = None
for idx, label in enumerate(block_labels):
    if label is None:
        continue
    if label != previous_label:
        block_markers.append({"index": idx, "text": _block_annotation_text(label)})
        previous_label = label

f"Loaded {x_split.shape[0]} timepoints × {x_split.shape[1]} neurons; {len(block_markers)} block boundaries"


In [ ]:
# Compute train/test split indices (fold 4 of 7, same as grid-search default)
n_samples = len(x_split) - split_window
_, test_idx = list(KFold(n_splits=7).split(range(n_samples)))[4]
test_start_index, test_end_index = int(test_idx[0]), int(test_idx[-1])

f"Test split: indices {test_start_index} – {test_end_index}"


In [ ]:
def plotting_neuronal_behavioural_with_split(
    x,
    b=None,
    b_names=None,
    b_colors=None,
    s=None,
    s_names=None,
    r=None,
    r_names=None,
    show_fig=True,
    test_split_start=None,
    test_split_end=None,
    block_markers=None,
    **kwargs,
):
    """Wrap plotting_neuronal_behavioural_plotly and overlay block + test-split markers.

    Extra args:
        test_split_start: raw time index where the test split begins (red dotted line).
        test_split_end:   raw time index where the test split ends   (red dotted line).
        block_markers:    list of dicts with keys ``index`` (int) and ``text`` (str).
    """
    num_plots = 1 + sum(1 for v in [b, s, r] if v is not None)

    fig = plotting_neuronal_behavioural_plotly(
        x=x,
        b=b,
        b_names=b_names,
        b_colors=b_colors,
        s=s,
        s_names=s_names,
        r=r,
        r_names=r_names,
        show_fig=False,
        **kwargs,
    )

    # --- block boundaries (green solid) ---
    if block_markers:
        for marker in block_markers:
            position = marker.get("index")
            if position is None:
                continue
            for row in range(1, num_plots + 1):
                fig.add_vline(
                    x=position, line_dash="solid", line_color="green",
                    line_width=1, row=row, col=1,
                )
            fig.add_annotation(
                x=position, y=1.08, xref="x1", yref="paper",
                text=marker.get("text", "Block"),
                showarrow=False, font=dict(color="green", size=11),
            )

    # --- test-split boundaries (red dotted) ---
    for idx, label in [(test_split_start, "test start"), (test_split_end, "test end")]:
        if idx is None:
            continue
        for row in range(1, num_plots + 1):
            fig.add_vline(
                x=idx, line_dash="dot", line_color="red",
                line_width=2, row=row, col=1,
            )
        fig.add_annotation(
            x=idx, y=1.02, xref="x1", yref="paper",
            text=label, showarrow=False, font=dict(color="red", size=12),
        )

    if show_fig:
        fig.show()

    return fig


In [ ]:
fig_split = plotting_neuronal_behavioural_with_split(
    x=x_split,
    b=b_split,
    b_names=b_labels_split,
    b_colors=b_colors_split,
    show_fig=True,
    test_split_start=test_start_index,
    test_split_end=test_end_index,
    block_markers=block_markers,
    colorscale="Viridis",
)

fig_split.write_html(output_html)
output_html


## HGF Belief Trajectory

Load the dataset with HGF belief alignment enabled (`hgf_model='binary2'`). This adds
`dataset.hgf_beliefs`: a per-timepoint array of the precision-weighted log-odds belief rescaled to [-1, 1]
(`x_1_expected_mean` ∈ [-1, 1]), forward-filled through intertrial gaps.

Two plots:
1. **Standalone belief trace** – with green block-boundary lines to verify the belief tracks reward context.
2. **Neuronal heatmap + belief trace** – stacked subplots sharing the x-axis.

In [ ]:
# Load the same session with HGF belief alignment (binary2 = 2-level hierarchy).
# recompute_cache=True forces recompute with updated hgf_model and normalization.
dataset_hgf = BanditTaskNeuroPixelsDataset(
    data_path=session_path,
    downsample_fs=30,
    downsample_method='count',
    good_neurons_only=True,
    normalize_method=None,
    choosing_state_mode='correctness',
    hgf_model='binary2'
)

hgf_beliefs = dataset_hgf.hgf_beliefs
assert hgf_beliefs is not None, 'hgf_beliefs is None - check hgf_model parameter'
assert len(hgf_beliefs) == dataset_hgf.b.shape[1], 'length mismatch'

print(f'hgf_beliefs  shape : {hgf_beliefs.shape}')
print(f'value range        : [{hgf_beliefs.min():.3f}, {hgf_beliefs.max():.3f}]')
print(f'mean +/- std       : {hgf_beliefs.mean():.3f} +/- {hgf_beliefs.std():.3f}')


In [ ]:
import plotly.graph_objects as go

timepoints = np.arange(len(hgf_beliefs))
block_lbl = np.array(dataset_hgf.block_labels, dtype=object)

fig_belief = go.Figure()
fig_belief.add_trace(go.Scatter(
    x=timepoints,
    y=hgf_beliefs,
    mode='lines',
    name='x_1_expected_mean',
    line=dict(color='steelblue', width=1.2),
))

# Overlay block boundaries
prev_lbl = None
for idx, lbl in enumerate(block_lbl):
    if lbl is not None and lbl != prev_lbl:
        fig_belief.add_vline(x=idx, line_dash='solid', line_color='green', line_width=1.5)
        label_str = str(lbl)
        short = 'Better L' if 'left' in label_str.lower() else ('Better R' if 'right' in label_str.lower() else label_str)
        fig_belief.add_annotation(
            x=idx, y=1.05, xref='x', yref='paper',
            text=short, showarrow=False, font=dict(color='green', size=10),
        )
        prev_lbl = lbl

fig_belief.update_layout(
    title='HGF Belief Trajectory (binary2, x_1_expected_mean (rescaled to [-1,1])) with Block Boundaries',
    xaxis_title='Timepoint (30 Hz)',
    yaxis_title='Belief (rescaled to [-1, 1])',
    yaxis=dict(range=[-1.05, 1.05]),
    height=350,
    hovermode='x unified',
)
fig_belief.show()


In [ ]:
from plotly.subplots import make_subplots

x_hgf = dataset_hgf.x.toarray()  # (neurons, T)

fig_comb = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[3, 1],
    subplot_titles=['Neuronal activation', 'HGF belief (x_1_expected_mean, rescaled to [-1,1])'],
)

# Neuronal heatmap
fig_comb.add_trace(
    go.Heatmap(z=x_hgf, colorscale='Viridis', showscale=False, name='neurons'),
    row=1, col=1,
)

# HGF belief line
fig_comb.add_trace(
    go.Scatter(
        x=timepoints, y=hgf_beliefs,
        mode='lines', name='x_1_expected_mean',
        line=dict(color='steelblue', width=1.2),
    ),
    row=2, col=1,
)

# Block boundaries on both rows
prev_lbl = None
for idx, lbl in enumerate(block_lbl):
    if lbl is not None and lbl != prev_lbl:
        for row in [1, 2]:
            fig_comb.add_vline(x=idx, line_dash='solid', line_color='green', line_width=1, row=row, col=1)
        prev_lbl = lbl

fig_comb.update_yaxes(title_text='Neuron index', row=1, col=1)
fig_comb.update_yaxes(title_text='Belief [-1,1]', range=[-1.05, 1.05], row=2, col=1)
fig_comb.update_xaxes(title_text='Timepoint (30 Hz)', row=2, col=1)
fig_comb.update_layout(
    title='Neuronal Activity + HGF Belief Trajectory',
    height=600,
    hovermode='x unified',
)
fig_comb.show()
